In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch

def predict_tariff_xlsx(input_file_path=None, output_file_path=None, model_path='Bartenderr/Bert_snomed'):
    """
    In this function we will collect the input file as xlsx and output same
    format.

    Inbetween we will try to get the snomed code for each tariff name in our file
    using a finetuned Bio_ClinicalBERT model

    to-do -- separate this into a different work book
    to-do -- retraining to improve the model.
    """
    #Just some Data prep
    tariff_df = pd.read_excel('tariff_df.xlsx')
    tariff_df['snomed code'] = tariff_df['snomed code'].astype(str)
    tariff_df['snomed code'] =tariff_df['snomed code'].str.replace('.0', '', regex=False)
    tariff_df.head()

    label_encoder = LabelEncoder()
    label_encoder.fit(tariff_df['snomed code'].unique())

    #Making this dynamic
    if input_file_path is None:
        input_file_path = input("Enter the input file path : ").strip()

    if output_file_path is None:
        output_file_path = input("Enter the output file path : ").strip()

    # Set this is up to use available device, cpu if gpu is-not-availble
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_path, 
                                                               output_attentions=True, 
                                                               attn_implementation="eager")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model.to(device)
    model.eval()

    # Load mapping from code to description
    #create a csv that we can update with new training
    snomed_set = tariff_df[['snomed code', 'snomed description']].drop_duplicates()
    code_to_desc = snomed_set.set_index('snomed code')['snomed description'].to_dict()

    # Load my input excel here
    df = pd.read_excel(input_file_path)

    # Error handling - Verify required column exists
    if 'TARIFF NAME' not in df.columns:
        raise ValueError("Input file must contain 'TARIFF NAME' column")

    
    df['TARIFF NAME'] = df['TARIFF NAME'].fillna('').astype(str)

    # init an empty list for predictions
    predictions = []
    descriptions = []
    confidence_scores = []
    attention_weights = []

    for text in df['TARIFF NAME']:
        try:
            
            if not text.strip():
                predictions.append("INVALID_INPUT")
                descriptions.append("Empty input text")
                confidence_scores.append(0)
                attention_weights.append("")
                continue

            inputs = tokenizer(
                text,
                padding='max_length',
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():
                outputs = model(**inputs)

            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            pred_class = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred_class].item()
            pred_label = label_encoder.inverse_transform([pred_class])[0]
            pred_desc = code_to_desc.get(pred_label, "Description not found")
            
            
            #attention weight and token
            attentions = torch.stack(outputs.attentions)  # [layers, batch, heads, seq_len, seq_len]
            avg_attention = attentions.mean(0).mean(1).squeeze()  # Average across layers and heads
            important_tokens = avg_attention.mean(dim=0)  # Average attention per token
            
            # Get top 5 most attended tokens
            token_ids = inputs['input_ids'][0]
            top_indices = important_tokens.argsort(descending=True)[:5]
            top_tokens = [(tokenizer.decode([token_ids[i].item()]), 
                         important_tokens[i].item()) 
                         for i in top_indices if i < len(token_ids)]
            
            attention_str = "; ".join([f"{token[0]}({token[1]:.2f})" for token in top_tokens])

            predictions.append(pred_label)
            descriptions.append(pred_desc)
            confidence_scores.append(confidence)
            attention_weights.append(attention_str)

        except Exception as e:
            print(f"Error processing text: {text[:50]}... Error: {str(e)}")
            predictions.append("PREDICTION_ERROR")
            descriptions.append("Error during processing")
            confidence_scores.append(0)
            attention_weights.append("")
            
            
    # Prediction dataframe
    df['Predicted Code'] = predictions
    df['Predicted Description'] = descriptions
    df['Confidence Score'] = confidence_scores
    df['Top Attention Tokens'] = attention_weights

    # Save results
    df.to_excel(output_file_path, index=True)
    print(f"Predictions saved to {output_file_path}")


#call my function nicely
predict_tariff_xlsx()